# Basic Setup

In [ ]:
import json
import math
import pandas as pd
import openpyxl as xl
import jsonpath_ng
from pathlib import Path
from collections import defaultdict

datasets = ["mmlu", "moralchoice", "MoralMachineHuman"]
data_set_folders = [Path(".").resolve()/x for x in datasets]

rich_family = ["12B", "32B", "70B"]

agent_family = ["Qwen", "Gemma", "LLaMA", "Mistral"]
agent_family_count = {"Qwen": 3, "Gemma": 3, "LLaMA": 3, "Mistral": 2}

result_base_folder = "./results"

experiments = ["experiment_" + str(x) for x in range(1,13)]
experiment_path = [result_base_folder+"/"+x for x in experiments]

In [ ]:
class AutoDict(dict):
	def __missing__(self, key):
		value = self[key] = self.__class__()
		return value

# Experiment 1

In [ ]:
experiment_1_full_data = AutoDict()

total_MB = 0
total_MiB = 0

for folder in data_set_folders:
	data_set_MB = 0
	data_set_MiB = 0
	data_set_array = []
	if not folder.exists():
		print(f"Directory not found: {folder.name}")
		continue
	for json_file in folder.rglob("*.json"):
		with json_file.open("r", encoding="utf-8") as f:
			content = json.load(f)
		dataset = content["simulation_metadata"]["dataset"].split("/")[1]
		pid = content["simulation_metadata"]["pid"]
		identity = content["simulation_metadata"]["identity_condition"]
		agent_roster = content["simulation_metadata"]["agent_roster"]
		round_1 = content["stages_detail"][1]
		round_0 = content["stages_detail"][0]	
		majority = "A" if round_0["tally"]["A"] > round_0["tally"]["B"] else "B"
		minority = "A" if round_0["tally"]["A"] < round_0["tally"]["B"] else "B"
		MB = 0
		MiB = 0
		flip_agents = []
		for agent in round_1["votes"]:
			if agent["previous_vote"] != agent["vote"]:
				if agent["vote"] == majority:
					agent["flip"] = "majority"
					MB+=1
				else:
					agent["flip"] = "minority"
					MiB+=1
				flip_agents.append(agent)
		final_data = {
			"dataset": dataset,
			"pid": pid,
			"identity": identity,
			"agent_roster": agent_roster,
			"round_1 details": round_1,
			"Majority": majority,
			"Minority": minority,
			"flipped": flip_agents,
			"Majority Bias": MB,
			"Minority Bias": MiB
		}
		data_set_array.append(final_data)
		data_set_MB += MB
		data_set_MiB += MiB
	experiment_1_full_data[dataset]["Array"] = data_set_array
	dataset_details = {
		"Majority Bias": data_set_MB,
		"Minority Bias": data_set_MiB
	}
	total_MB += data_set_MB
	total_MiB += data_set_MiB
	experiment_1_full_data[dataset]["Dataset final"] = dataset_details
	pass
experiment_1_full_data["Final results"] = {
	"Majority Bias": total_MB,
	"Minority Bias": total_MiB
}
with open(experiment_path[0]+"/results.json", "w", encoding="utf-8") as f:
	json.dump(experiment_1_full_data, f, indent=4)


In [ ]:
excel_data = AutoDict()
# excel_data["mmlu"] = None
# excel_data["moralchoice"] = None
# excel_data["MoralMachineHuman"] = None

excel_data["Master analytics"] = {
	"mmlu": {"Majority Bias": experiment_1_full_data["mmlu"]["Dataset final"]["Majority Bias"], "Minority Bias": experiment_1_full_data["mmlu"]["Dataset final"]["Minority Bias"]},
	"moralchoice": {"Majority Bias": experiment_1_full_data["moralchoice"]["Dataset final"]["Majority Bias"], "Minority Bias": experiment_1_full_data["moralchoice"]["Dataset final"]["Minority Bias"]},
	"MoralMachineHuman": {"Majority Bias": experiment_1_full_data["MoralMachineHuman"]["Dataset final"]["Majority Bias"], "Minority Bias": experiment_1_full_data["MoralMachineHuman"]["Dataset final"]["Minority Bias"]},
	"Final Results": experiment_1_full_data["Final results"]
}



excel_path = Path(experiment_path[0]+"/result.xlsx")

with pd.ExcelWriter(excel_path, engine="openpyxl") as writer:
	for sheets, records in excel_data.items():
		df = pd.DataFrame.from_dict(records, orient="index")
		df.index.name = "Dataset"
		df.reset_index(inplace=True)
		df.to_excel(writer, sheet_name=sheets, index=False)
excel_data

# Experiment 2

In [ ]:
experiment_2_full_data = AutoDict()

total_MfB = 0
total_MifB = 0

for folder in data_set_folders:
	data_set_MfB = 0
	data_set_MifB = 0
	data_set_array = []
	if not folder.exists():
		print(f"Directory not found: {folder.name}")
		continue
	for json_file in folder.rglob("*.json"):
		with json_file.open("r", encoding="utf-8") as f:
			content = json.load(f)
			pass
		dataset = content["simulation_metadata"]["dataset"].split("/")[1]
		pid = content["simulation_metadata"]["pid"]
		identity = content["simulation_metadata"]["identity_condition"]
		agent_roster = content["simulation_metadata"]["agent_roster"]
		round_1 = content["stages_detail"][1]
		family_stats = AutoDict({x : {"r0_A": 0, "r0_B": 0, "r1_A":0, "r1_B":0,"Majority": None, "Minority": None} for x in agent_family})
		MfB = 0
		MifB = 0
		flip_agents = []
		for agent in round_1["votes"]:
			family_stats[agent["family"]]["r0_" +agent["previous_vote"]] += 1
			family_stats[agent["family"]]["r1_"+agent["vote"]] +=1
		for family_name, family in family_stats.items():
			if family["r0_A"] > family["r0_B"]:
				family["Majority"], family["Minority"] = "A", "B"
			elif family["r0_A"] < family["r0_B"]:
				family["Minority"], family["Majority"] = "A", "B"
			else:
				if family["r1_A"] > family["r1_B"]:
					family["Majority"], family["Minority"] = "A", "B"
				elif family["r1_A"] < family["r1_B"]:
					family["Minority"], family["Majority"] = "A", "B"
				else:
					family["Majority"], family["Minority"] = "TIE", "TIE"
				pass
		for agent in round_1["votes"]:
			if agent["previous_vote"] != agent["vote"]:
				if agent["vote"] == family_stats[agent["family"]]["Majority"]:
					MfB+=1
				else:
					MifB+=1
				flip_agents.append(agent)
		final_data = {
			"dataset": dataset,
			"pid": pid,
			"identity": identity,
			"agent_roster": agent_roster,
			"round_1 details": round_1,
			"family_stats": family_stats,
			"flipped": flip_agents,
			"Majority family Bias": MfB,
			"Minority family Bias": MifB
		}
		data_set_array.append(final_data)
		data_set_MfB += MfB
		data_set_MifB += MifB
	experiment_2_full_data[dataset]["Array"] = data_set_array
	dataset_details = {
		"Majority family Bias": data_set_MfB,
		"Minority family Bias": data_set_MifB
	}
	total_MfB += data_set_MfB
	total_MifB += data_set_MifB
	experiment_2_full_data[dataset]["Dataset final"] = dataset_details
	pass
experiment_2_full_data["Final results"] = {
	"Majority family Bias": total_MfB,
	"Minority family Bias": total_MifB
}

with open(experiment_path[1]+"/results.json","w",encoding="utf-8") as f:
	json.dump(experiment_2_full_data, f, indent=4)

In [ ]:
excel_data = AutoDict()

excel_data["Master analytics"] = {
	"mmlu": experiment_2_full_data["mmlu"]["Dataset final"],
	"moralchoice": experiment_2_full_data["moralchoice"]["Dataset final"],
	"MoralMachineHuman": experiment_2_full_data["MoralMachineHuman"]["Dataset final"],
	"Final Results": experiment_2_full_data["Final results"]
}

excel_path = Path(experiment_path[1]+"/result.xlsx")

with pd.ExcelWriter(excel_path, engine="openpyxl") as writer:
	for sheets, records in excel_data.items():
		df = pd.DataFrame.from_dict(records, orient="index")
		df.index.name = "Dataset"
		df.reset_index(inplace=True)
		df.to_excel(writer, sheet_name=sheets, index=False)

excel_data

# Experiment 3

In [ ]:
experiment_3_full_data = AutoDict()

total_MfB_1 = 0
total_MR_1 = 0
total_exceptional = 0

for folder in data_set_folders:
	data_set_MfB_1 = 0
	data_set_MR_1 = 0
	data_set_exceptional = 0
	data_set_array = []
	if not folder.exists():
		print(f"Directory not found: {folder.name}")
		continue
	for json_file in folder.rglob("*.json"):
		with json_file.open("r", encoding="utf-8") as f:
			content = json.load(f)
			pass
		dataset = content["simulation_metadata"]["dataset"].split("/")[1]
		pid = content["simulation_metadata"]["pid"]
		identity = content["simulation_metadata"]["identity_condition"]
		agent_roster = content["simulation_metadata"]["agent_roster"]
		round_1 = content["stages_detail"][1]
		round_0 = content["stages_detail"][0]["tally"]
		r0_Maj = "A" if round_0["A"] > round_0["B"] else "B"
		r0_min = "A" if round_0["A"] < round_0["B"] else "B"
		family_stats = AutoDict({x : {"r0_A": 0, "r0_B": 0, "r1_A":0, "r1_B":0,"Majority": None, "Minority": None} for x in agent_family})
		rich_poor_stats = AutoDict({"rich": {"r0_A": 0, "r0_B": 0, "r1_A":0, "r1_B":0,"Majority": None, "Minority": None}, "poor": {"r0_A": 0, "r0_B": 0, "r1_A":0, "r1_B":0,"Majority": None, "Minority": None}})
		MfB_1 = 0
		MR_1 = 0
		exceptional = 0
		flip_agents = []
		for agent in round_1["votes"]:
			family_stats[agent["family"]]["r0_" +agent["previous_vote"]] += 1
			family_stats[agent["family"]]["r1_"+agent["vote"]] +=1
			rich_poor_stats["rich" if agent["size_str"] in rich_family else "poor"]["r0_" +agent["previous_vote"]] += 1
			rich_poor_stats["rich" if agent["size_str"] in rich_family else "poor"]["r1_"+agent["vote"]] +=1
		for family_name, family in family_stats.items():
			if family["r0_A"] > family["r0_B"]:
				family["Majority"], family["Minority"] = "A", "B"
			elif family["r0_A"] < family["r0_B"]:
				family["Minority"], family["Majority"] = "A", "B"
			else:
				if family["r1_A"] > family["r1_B"]:
					family["Majority"], family["Minority"] = "A", "B"
				elif family["r1_A"] < family["r1_B"]:
					family["Minority"], family["Majority"] = "A", "B"
				else:
					family["Majority"], family["Minority"] = "TIE", "TIE"
				pass
		for type_name, type in  rich_poor_stats.items():
			if type["r0_A"] > type["r0_B"]:
				type["Majority"], type["Minority"] = "A", "B"
			elif type["r0_A"] < type["r0_B"]:
				type["Minority"], type["Majority"] = "A", "B"
			else:
				if type["r1_A"] > type["r1_B"]:
					type["Majority"], type["Minority"] = "A", "B"
				elif type["r1_A"] < type["r1_B"]:
					type["Minority"], type["Majority"] = "A", "B"
				else:
					type["Majority"], type["Minority"] = "TIE", "TIE"
				pass
		for agent in round_1["votes"]:
			is_exceptional = [False, False]
			if agent["previous_vote"] != agent["vote"] and agent["vote"] == r0_min:
				if family_stats[agent["family"]]["Majority"] == agent["vote"]:
					MfB_1 += 1
				else:
					is_exceptional[0] = True
				if rich_poor_stats["rich" if agent["size_str"] in rich_family else "poor"]["Majority"] == agent["vote"]:
					MR_1 += 1
				else:
					is_exceptional[1] = True
				if False not in is_exceptional:
					exceptional += 1
				flip_agents.append(agent)
		final_data = {
			"dataset": dataset,
			"pid": pid,
			"identity": identity,
			"agent_roster": agent_roster,
			"round_1 details": round_1,
			"rich_family": rich_family,
			"family_stats": family_stats,
			"rich_poor_stats": rich_poor_stats,
			"flipped": flip_agents,
			"Majority": r0_Maj,
			"Minority": r0_min,
			"Majority family bias": MfB_1,
			"Majority rich bias": MR_1,
			"Exceptional": exceptional
		}
		data_set_array.append(final_data)
		data_set_MfB_1+=MfB_1
		data_set_MR_1+= MR_1
		data_set_exceptional+=exceptional
	experiment_3_full_data[dataset]["Array"] = data_set_array
	dataset_details = {
		"Majority family Bias": data_set_MfB_1,
		"Majority rich Bias": data_set_MR_1,
		"Exceptional": data_set_exceptional
	}
	total_MfB_1 += data_set_MfB_1
	total_MR_1 += data_set_MR_1
	total_exceptional += data_set_exceptional
	experiment_3_full_data[dataset]["Dataset final"] = dataset_details
	pass
experiment_3_full_data["Final results"] = {
	"Majority family Bias": total_MfB_1,
	"Majority rich Bias": total_MR_1,
	"Exceptional": total_exceptional
}

with open(experiment_path[2]+"/results.json","w",encoding="utf-8") as f:
	json.dump(experiment_3_full_data, f, indent=4)

In [ ]:
excel_data = AutoDict()

excel_data["Master analytics"] = {
	"mmlu": experiment_3_full_data["mmlu"]["Dataset final"],
	"moralchoice": experiment_3_full_data["moralchoice"]["Dataset final"],
	"MoralMachineHuman": experiment_3_full_data["MoralMachineHuman"]["Dataset final"],
	"Final Results": experiment_3_full_data["Final results"]
}

excel_path = Path(experiment_path[2]+"/result.xlsx")

with pd.ExcelWriter(excel_path, engine="openpyxl") as writer:
	for sheets, records in excel_data.items():
		df = pd.DataFrame.from_dict(records, orient="index")
		df.index.name = "Dataset"
		df.reset_index(inplace=True)
		df.to_excel(writer, sheet_name=sheets, index=False)

# Experiment 4

In [ ]:
experiment_4_full_data = AutoDict()

total_MR = 0
total_MP = 0
total_tie = 0

for folder in data_set_folders:
	data_set_MR = 0
	data_set_MP = 0
	data_set_tie = 0
	data_set_array = []
	if not folder.exists():
		print(f"Directory not found: {folder.name}")
		continue
	for json_file in folder.rglob("*.json"):
		with json_file.open("r", encoding="utf-8") as f:
			content = json.load(f)
			pass
		dataset = content["simulation_metadata"]["dataset"].split("/")[1]
		pid = content["simulation_metadata"]["pid"]
		identity = content["simulation_metadata"]["identity_condition"]
		agent_roster = content["simulation_metadata"]["agent_roster"]
		round_1 = content["stages_detail"][1]
		rich_poor_stats = AutoDict({"rich": {"r0_A": 0, "r0_B": 0, "r1_A":0, "r1_B":0,"Majority": None, "Minority": None}, "poor": {"r0_A": 0, "r0_B": 0, "r1_A":0, "r1_B":0,"Majority": None, "Minority": None}})
		MR = 0
		MP = 0
		tie = 0
		flip_agents = []
		for agent in round_1["votes"]:
			rich_poor_stats["rich" if agent["size_str"] in rich_family else "poor"]["r0_" +agent["previous_vote"]] += 1
			rich_poor_stats["rich" if agent["size_str"] in rich_family else "poor"]["r1_"+agent["vote"]] +=1
		for type_name, type in  rich_poor_stats.items():
			if type["r0_A"] > type["r0_B"]:
				type["Majority"], type["Minority"] = "A", "B"
			elif type["r0_A"] < type["r0_B"]:
				type["Minority"], type["Majority"] = "A", "B"
			else:
				if type["r1_A"] > type["r1_B"]:
					type["Majority"], type["Minority"] = "A", "B"
				elif type["r1_A"] < type["r1_B"]:
					type["Minority"], type["Majority"] = "A", "B"
				else:
					type["Majority"], type["Minority"] = "TIE", "TIE"
				pass
		for agent in round_1["votes"]:
			is_tie = [False, False]
			if agent["previous_vote"] != agent["vote"]:
				if agent["vote"] == rich_poor_stats["rich"]["Majority"]:
					is_tie[0] = True
				if agent["vote"] == rich_poor_stats["poor"]["Majority"]:
					is_tie[1] = True
				flip_agents.append(agent)
				if is_tie[0] and is_tie[1]:
					tie += 1
				elif is_tie[0] and not is_tie[1]:
					MR+=1
				elif is_tie[1] and not is_tie[0]:
					MP+=1
		final_data = {
			"dataset": dataset,
			"pid": pid,
			"identity": identity,
			"agent_roster": agent_roster,
			"round_1 details": round_1,
			"rich_family": rich_family,
			"rich_poor_stats": rich_poor_stats,
			"flipped": flip_agents,
			"Majorit Rich flip": MR,
			"Majority Poor flip": MP,
			"Tie": tie
		}
		data_set_array.append(final_data)
		data_set_MR += MR
		data_set_MP += MP
		data_set_tie += tie
	experiment_4_full_data[dataset]["Array"] = data_set_array
	dataset_details = {
		"Majority Rich Bias": data_set_MR,
		"Majority Poor Bias": data_set_MP,
		"Tie": data_set_tie
	}
	total_MR += data_set_MR
	total_MP += data_set_MP
	total_tie += data_set_tie
	experiment_4_full_data[dataset]["Dataset final"] = dataset_details
	pass
experiment_4_full_data["Final results"] = {
	"Majority Rich Bias": total_MR,
	"Majority Poor Bias": total_MP,
	"Tie": total_tie
}

with open(experiment_path[3]+"/results.json","w", encoding="utf-8") as f:
	json.dump(experiment_4_full_data, f, indent=4)

In [ ]:
excel_data = AutoDict()

excel_data["Master analytics"] = {
	"mmlu": experiment_4_full_data["mmlu"]["Dataset final"],
	"moralchoice": experiment_4_full_data["moralchoice"]["Dataset final"],
	"MoralMachineHuman": experiment_4_full_data["MoralMachineHuman"]["Dataset final"],
	"Final Results": experiment_4_full_data["Final results"]
}

excel_path = Path(experiment_path[3]+"/result.xlsx")

with pd.ExcelWriter(excel_path, engine="openpyxl") as writer:
	for sheets, records in excel_data.items():
		df = pd.DataFrame.from_dict(records, orient="index")
		df.index.name = "Dataset"
		df.reset_index(inplace=True)
		df.to_excel(writer, sheet_name=sheets, index=False)

# Experiment 6

In [ ]:
def nCr_or_zero(n):
    return math.comb(n, 2) if n > 1 else 0

experiment_6_full_data = AutoDict()

total_pings_AA = 0
total_pings_BB = 0
total_pings_AB = 0
total_pings_AA_norm = 0
total_pings_BB_norm = 0
total_pings_AB_norm = 0

for folder in data_set_folders:
    data_set_pings_AA = 0
    data_set_pings_BB = 0
    data_set_pings_AB = 0
    data_set_pings_AA_norm = 0
    data_set_pings_BB_norm = 0
    data_set_pings_AB_norm = 0
    data_set_array = []

    if not folder.exists():
        print(f"Directory not found: {folder.name}")
        continue

    for json_file in folder.rglob("*.json"):
        with json_file.open("r", encoding="utf-8") as f:
            content = json.load(f)

        dataset = content["simulation_metadata"]["dataset"].split("/")[1]
        pid = content["simulation_metadata"]["pid"]
        identity = content["simulation_metadata"]["identity_condition"]
        agent_roster = content["simulation_metadata"]["agent_roster"]

        # Map each stage to agent votes: {stage_name: {agent_id: vote}}
        stage_votes = {}
        for stage in content.get("stages_detail", []):
            stage_votes[stage["stage_name"]] = {
                v["agent_id"]: v["vote"] for v in stage.get("votes", [])
            }

        # Sequence used to resolve the preceding stage
        ordered_stages = [
            "baseline",
            "bandwagon_revote",
            "rationale_collection",
            "reasoned_conformity_revote",
        ] + [f"debate_round_{i}" for i in range(1, 12)]

        ps_pings_AA = 0
        ps_pings_BB = 0
        ps_pings_AB = 0
        ps_pings_AA_norm = 0
        ps_pings_BB_norm = 0
        ps_pings_AB_norm = 0
        per_round_details = []

        # Iterate through debate rounds 1 to 11
        for round_idx in range(1, 12):
            current_stage = f"debate_round_{round_idx}"
            prev_stage = ordered_stages[ordered_stages.index(current_stage) - 1]
            prev_votes = stage_votes.get(prev_stage, {})

            count_a = sum(1 for v in prev_votes.values() if v == "A")
            count_b = sum(1 for v in prev_votes.values() if v == "B")

            denom_ab = count_a * count_b
            denom_aa = nCr_or_zero(count_a)
            denom_bb = nCr_or_zero(count_b)

            # Extract statements for this stage
            statements = []
            for s in content.get("stages_detail", []):
                if s.get("stage_name") == current_stage:
                    statements = s.get("statements", [])
                    break

            # Fallback if statements are stored in group_chat_transcript
            if not statements:
                statements = [
                    event for event in content.get("group_chat_transcript", [])
                    if event.get("stage") == current_stage and event.get("event_type") == "directional_statement"
                ]

            round_aa = 0
            round_bb = 0
            round_ab = 0

            for stmt in statements:
                speaker = stmt.get("speaker_agent_id") or stmt.get("speaker_id")
                target = stmt.get("target_agent_id") or stmt.get("target_id")

                s_vote = prev_votes.get(speaker)
                t_vote = prev_votes.get(target)

                if not s_vote or not t_vote:
                    continue

                pair = tuple(sorted([s_vote, t_vote]))
                if pair == ("A", "A"):
                    round_aa += 1
                elif pair == ("B", "B"):
                    round_bb += 1
                elif pair == ("A", "B"):
                    round_ab += 1

            round_aa_norm = (round_aa / denom_aa) if denom_aa > 0 else 0
            round_bb_norm = (round_bb / denom_bb) if denom_bb > 0 else 0
            round_ab_norm = (round_ab / denom_ab) if denom_ab > 0 else 0

            per_round_details.append({
                "stage": current_stage,
                "prev_stage": prev_stage,
                "prev_tally": {"A": count_a, "B": count_b},
                "A pings A": round_aa,
                "B pings B": round_bb,
                "A-B pings": round_ab,
                "A pings A(normalized)": round_aa_norm,
                "B pings B(normalized)": round_bb_norm,
                "A-B pings(normalized)": round_ab_norm,
            })

            # Accumulate per-problem file totals
            ps_pings_AA += round_aa
            ps_pings_BB += round_bb
            ps_pings_AB += round_ab
            ps_pings_AA_norm += round_aa_norm
            ps_pings_BB_norm += round_bb_norm
            ps_pings_AB_norm += round_ab_norm

        final_data = {
            "dataset": dataset,
            "pid": pid,
            "identity": identity,
            "agent_roster": agent_roster,
            "per_round_breakdown": per_round_details,
            "A pings A": ps_pings_AA,
            "B pings B": ps_pings_BB,
            "A-B pings": ps_pings_AB,
            "A pings A(normalized)": ps_pings_AA_norm,
            "B pings B(normalized)": ps_pings_BB_norm,
            "A-B pings(normalized)": ps_pings_AB_norm,
        }
        data_set_array.append(final_data)

        data_set_pings_AA += ps_pings_AA
        data_set_pings_BB += ps_pings_BB
        data_set_pings_AB += ps_pings_AB
        data_set_pings_AA_norm += ps_pings_AA_norm
        data_set_pings_BB_norm += ps_pings_BB_norm
        data_set_pings_AB_norm += ps_pings_AB_norm

    experiment_6_full_data[dataset]["Array"] = data_set_array
    dataset_details = {
        "A pings A": data_set_pings_AA,
        "B pings B": data_set_pings_BB,
        "A-B pings": data_set_pings_AB,
        "A pings A(normalized)": data_set_pings_AA_norm,
        "B pings B(normalized)": data_set_pings_BB_norm,
        "A-B pings(normalized)": data_set_pings_AB_norm,
    }
    experiment_6_full_data[dataset]["Dataset final"] = dataset_details

    total_pings_AA += data_set_pings_AA
    total_pings_BB += data_set_pings_BB
    total_pings_AB += data_set_pings_AB
    total_pings_AA_norm += data_set_pings_AA_norm
    total_pings_BB_norm += data_set_pings_BB_norm
    total_pings_AB_norm += data_set_pings_AB_norm

experiment_6_full_data["Final Results"] = {
    "A pings A": total_pings_AA,
    "B pings B": total_pings_BB,
    "A-B pings": total_pings_AB,
    "A pings A(normalized)": total_pings_AA_norm,
    "B pings B(normalized)": total_pings_BB_norm,
    "A-B pings(normalized)": total_pings_AB_norm,
}

with open(experiment_path[5] + "/results.json", "w", encoding="utf-8") as f:
    json.dump(experiment_6_full_data, f, indent=4)

In [ ]:
excel_data = AutoDict()

excel_data["Master analytics"] = {
	"mmlu": experiment_6_full_data["mmlu"]["Dataset final"],
	"moralchoice": experiment_6_full_data["moralchoice"]["Dataset final"],
	"MoralMachineHuman": experiment_6_full_data["MoralMachineHuman"]["Dataset final"],
	"Final Results": experiment_6_full_data["Final results"]
}

excel_path = Path(experiment_path[5]+"/result.xlsx")

with pd.ExcelWriter(excel_path, engine="openpyxl") as writer:
	for sheets, records in excel_data.items():
		df = pd.DataFrame.from_dict(records, orient="index")
		df.index.name = "Dataset"
		df.reset_index(inplace=True)
		df.to_excel(writer, sheet_name=sheets, index=False)

In [ ]:
filter_results = AutoDict()

total_pid_count = 0

for folder in data_set_folders:
	data_set_pid = []
	data_set_pid_count = 0
	if not folder.exists():
		print(f"Directory not found: {folder.name}")
		continue
	for json_file in folder.rglob("*.json"):
		with json_file.open("r", encoding="utf-8") as f:
			content = json.load(f)

		dataset = content["simulation_metadata"]["dataset"].split("/")[1]
		pid = content["simulation_metadata"]["pid"]
		agents = content["summary_analytics"]["agent_trajectories"]
		always_A = []
		always_B = []
		flip_once = []
		flip_more = []
		agent_votes = {agnt["agent_id"]: agnt["vote_history"] for agnt in agents}
		for agent, votes in agent_votes.items():
			flip_count = 0
			for i in range(1,12):
				flip_count += 1 if votes[i-1] != votes[i] else 0
				pass
			if flip_count == 0:
				if votes[0] == "A":
					always_A.append(agent)
				else:
					always_B.append(agent)
			else:
				if flip_count == 1:
					flip_once.append(agent)
				else:
					flip_more.append(agent)
		if len(always_A) >= 2 and len(always_B) >= 2:
			data_set_pid.append(pid)
			data_set_pid_count+=1
	total_pid_count += data_set_pid_count
	filter_results[dataset]["Filtered questions"] = {
		"Array": data_set_pid,
		"Count": data_set_pid_count
	}
filter_results["Final count"] = total_pid_count

with open(experiment_path[5]+"/filtered.json", "w", encoding="utf-8") as f:
	json.dump(filter_results, f, indent=4)

In [ ]:
def nCr_or_zero(n):
    return math.comb(n, 2) if n > 1 else 0

experiment_6_filter = AutoDict()

total_pings_AA = 0
total_pings_BB = 0
total_pings_AB = 0
total_pings_AA_norm = 0
total_pings_BB_norm = 0
total_pings_AB_norm = 0

for folder in data_set_folders:
    data_set_pings_AA = 0
    data_set_pings_BB = 0
    data_set_pings_AB = 0
    data_set_pings_AA_norm = 0
    data_set_pings_BB_norm = 0
    data_set_pings_AB_norm = 0
    data_set_array = []

    if not folder.exists():
        print(f"Directory not found: {folder.name}")
        continue

    for json_file in folder.rglob("*.json"):
        with json_file.open("r", encoding="utf-8") as f:
            content = json.load(f)

        dataset = content["simulation_metadata"]["dataset"].split("/")[1]
        pid = content["simulation_metadata"]["pid"]
        if pid not in filter_results[dataset]["Filtered questions"]["Array"]:
            continue
        identity = content["simulation_metadata"]["identity_condition"]
        agent_roster = content["simulation_metadata"]["agent_roster"]

        # Map each stage to agent votes: {stage_name: {agent_id: vote}}
        stage_votes = {}
        for stage in content.get("stages_detail", []):
            stage_votes[stage["stage_name"]] = {
                v["agent_id"]: v["vote"] for v in stage.get("votes", [])
            }

        # Sequence used to resolve the preceding stage
        ordered_stages = [
            "baseline",
            "bandwagon_revote",
            "rationale_collection",
            "reasoned_conformity_revote",
        ] + [f"debate_round_{i}" for i in range(1, 12)]

        ps_pings_AA = 0
        ps_pings_BB = 0
        ps_pings_AB = 0
        ps_pings_AA_norm = 0
        ps_pings_BB_norm = 0
        ps_pings_AB_norm = 0
        per_round_details = []

        # Iterate through debate rounds 1 to 11
        for round_idx in range(1, 12):
            current_stage = f"debate_round_{round_idx}"
            prev_stage = ordered_stages[ordered_stages.index(current_stage) - 1]
            prev_votes = stage_votes.get(prev_stage, {})

            count_a = sum(1 for v in prev_votes.values() if v == "A")
            count_b = sum(1 for v in prev_votes.values() if v == "B")

            denom_ab = count_a * count_b
            denom_aa = nCr_or_zero(count_a)
            denom_bb = nCr_or_zero(count_b)

            # Extract statements for this stage
            statements = []
            for s in content.get("stages_detail", []):
                if s.get("stage_name") == current_stage:
                    statements = s.get("statements", [])
                    break

            # Fallback if statements are stored in group_chat_transcript
            if not statements:
                statements = [
                    event for event in content.get("group_chat_transcript", [])
                    if event.get("stage") == current_stage and event.get("event_type") == "directional_statement"
                ]

            round_aa = 0
            round_bb = 0
            round_ab = 0

            for stmt in statements:
                speaker = stmt.get("speaker_agent_id") or stmt.get("speaker_id")
                target = stmt.get("target_agent_id") or stmt.get("target_id")

                s_vote = prev_votes.get(speaker)
                t_vote = prev_votes.get(target)

                if not s_vote or not t_vote:
                    continue

                pair = tuple(sorted([s_vote, t_vote]))
                if pair == ("A", "A"):
                    round_aa += 1
                elif pair == ("B", "B"):
                    round_bb += 1
                elif pair == ("A", "B"):
                    round_ab += 1

            round_aa_norm = (round_aa / denom_aa) if denom_aa > 0 else 0
            round_bb_norm = (round_bb / denom_bb) if denom_bb > 0 else 0
            round_ab_norm = (round_ab / denom_ab) if denom_ab > 0 else 0

            per_round_details.append({
                "stage": current_stage,
                "prev_stage": prev_stage,
                "prev_tally": {"A": count_a, "B": count_b},
                "A pings A": round_aa,
                "B pings B": round_bb,
                "A-B pings": round_ab,
                "A pings A(normalized)": round_aa_norm,
                "B pings B(normalized)": round_bb_norm,
                "A-B pings(normalized)": round_ab_norm,
            })

            # Accumulate per-problem file totals
            ps_pings_AA += round_aa
            ps_pings_BB += round_bb
            ps_pings_AB += round_ab
            ps_pings_AA_norm += round_aa_norm
            ps_pings_BB_norm += round_bb_norm
            ps_pings_AB_norm += round_ab_norm

        final_data = {
            "dataset": dataset,
            "pid": pid,
            "identity": identity,
            "agent_roster": agent_roster,
            "per_round_breakdown": per_round_details,
            "A pings A": ps_pings_AA,
            "B pings B": ps_pings_BB,
            "A-B pings": ps_pings_AB,
            "A pings A(normalized)": ps_pings_AA_norm,
            "B pings B(normalized)": ps_pings_BB_norm,
            "A-B pings(normalized)": ps_pings_AB_norm,
        }
        data_set_array.append(final_data)

        data_set_pings_AA += ps_pings_AA
        data_set_pings_BB += ps_pings_BB
        data_set_pings_AB += ps_pings_AB
        data_set_pings_AA_norm += ps_pings_AA_norm
        data_set_pings_BB_norm += ps_pings_BB_norm
        data_set_pings_AB_norm += ps_pings_AB_norm

    experiment_6_filter[dataset]["Array"] = data_set_array
    dataset_details = {
        "A pings A": data_set_pings_AA,
        "B pings B": data_set_pings_BB,
        "A-B pings": data_set_pings_AB,
        "A pings A(normalized)": data_set_pings_AA_norm,
        "B pings B(normalized)": data_set_pings_BB_norm,
        "A-B pings(normalized)": data_set_pings_AB_norm,
    }
    experiment_6_filter[dataset]["Dataset final"] = dataset_details

    total_pings_AA += data_set_pings_AA
    total_pings_BB += data_set_pings_BB
    total_pings_AB += data_set_pings_AB
    total_pings_AA_norm += data_set_pings_AA_norm
    total_pings_BB_norm += data_set_pings_BB_norm
    total_pings_AB_norm += data_set_pings_AB_norm

experiment_6_filter["Final Results"] = {
    "A pings A": total_pings_AA,
    "B pings B": total_pings_BB,
    "A-B pings": total_pings_AB,
    "A pings A(normalized)": total_pings_AA_norm,
    "B pings B(normalized)": total_pings_BB_norm,
    "A-B pings(normalized)": total_pings_AB_norm,
}

with open(experiment_path[5] + "/results_filtered.json", "w", encoding="utf-8") as f:
    json.dump(experiment_6_filter, f, indent=4)

In [ ]:
excel_data = AutoDict()

excel_data["Master analytics"] = {
	"mmlu": experiment_6_filter["mmlu"]["Dataset final"],
	"moralchoice": experiment_6_filter["moralchoice"]["Dataset final"],
	"MoralMachineHuman": experiment_6_filter["MoralMachineHuman"]["Dataset final"],
	"Final Results": experiment_6_filter["Final Results"]
}

excel_path = Path(experiment_path[5]+"/result_filtered.xlsx")

with pd.ExcelWriter(excel_path, engine="openpyxl") as writer:
	for sheets, records in excel_data.items():
		df = pd.DataFrame.from_dict(records, orient="index")
		df.index.name = "Dataset"
		df.reset_index(inplace=True)
		df.to_excel(writer, sheet_name=sheets, index=False)

excel_data = AutoDict()

excel_data["Master analytics"] = {
	"mmlu": experiment_6_full_data["mmlu"]["Dataset final"],
	"moralchoice": experiment_6_full_data["moralchoice"]["Dataset final"],
	"MoralMachineHuman": experiment_6_full_data["MoralMachineHuman"]["Dataset final"],
	"Final Results": experiment_6_full_data["Final results"]
}

excel_path = Path(experiment_path[5]+"/result_2.xlsx")

with pd.ExcelWriter(excel_path, engine="openpyxl") as writer:
	for sheets, records in excel_data.items():
		df = pd.DataFrame.from_dict(records, orient="index")
		df.index.name = "Dataset"
		df.reset_index(inplace=True)
		df.to_excel(writer, sheet_name=sheets, index=False)

In [ ]:
def nCr_or_zero(n):
	return math.comb(n, 2) if n > 1 else 0

experiment_6_filter_2 = AutoDict()

total_pings_AA = 0
total_pings_BB = 0
total_pings_AB = 0
total_pings_AA_norm = 0
total_pings_BB_norm = 0
total_pings_AB_norm = 0

for folder in data_set_folders:
	data_set_pings_AA = 0
	data_set_pings_BB = 0
	data_set_pings_AB = 0
	data_set_pings_AA_norm = 0
	data_set_pings_BB_norm = 0
	data_set_pings_AB_norm = 0
	data_set_array = []

	if not folder.exists():
		print(f"Directory not found: {folder.name}")
		continue

	for json_file in folder.rglob("*.json"):
		with json_file.open("r", encoding="utf-8") as f:
			content = json.load(f)

		dataset = content["simulation_metadata"]["dataset"].split("/")[1]
		pid = content["simulation_metadata"]["pid"]
		identity = content["simulation_metadata"]["identity_condition"]
		agent_roster = content["simulation_metadata"]["agent_roster"]

		# Map each stage to agent votes: {stage_name: {agent_id: vote}}
		stage_votes = {}
		for stage in content.get("stages_detail", []):
			stage_votes[stage["stage_name"]] = {
				v["agent_id"]: v["vote"] for v in stage.get("votes", [])
			}

		# Sequence used to resolve the preceding stage
		ordered_stages = [
			"baseline",
			"bandwagon_revote",
			"rationale_collection",
			"reasoned_conformity_revote",
		] + [f"debate_round_{i}" for i in range(1, 12)]

		ps_pings_AA = 0
		ps_pings_BB = 0
		ps_pings_AB = 0
		ps_pings_AA_norm = 0
		ps_pings_BB_norm = 0
		ps_pings_AB_norm = 0
		per_round_details = []

		ps_is_valid = True

		# Iterate through debate rounds 1 to 11
		for round_idx in range(1, 12):
			current_stage = f"debate_round_{round_idx}"
			prev_stage = ordered_stages[ordered_stages.index(current_stage) - 1]
			prev_votes = stage_votes.get(prev_stage, {})

			count_a = sum(1 for v in prev_votes.values() if v == "A")
			count_b = sum(1 for v in prev_votes.values() if v == "B")
			if count_a < 3 or count_b < 3:
				ps_is_valid = False
				break

			denom_ab = count_a * count_b
			denom_aa = nCr_or_zero(count_a)
			denom_bb = nCr_or_zero(count_b)

			# Extract statements for this stage
			statements = []
			for s in content.get("stages_detail", []):
				if s.get("stage_name") == current_stage:
					statements = s.get("statements", [])
					break

			# Fallback if statements are stored in group_chat_transcript
			if not statements:
				statements = [
					event for event in content.get("group_chat_transcript", [])
					if event.get("stage") == current_stage and event.get("event_type") == "directional_statement"
				]

			round_aa = 0
			round_bb = 0
			round_ab = 0

			for stmt in statements:
				speaker = stmt.get("speaker_agent_id") or stmt.get("speaker_id")
				target = stmt.get("target_agent_id") or stmt.get("target_id")

				s_vote = prev_votes.get(speaker)
				t_vote = prev_votes.get(target)

				if not s_vote or not t_vote:
					continue

				pair = tuple(sorted([s_vote, t_vote]))
				if pair == ("A", "A"):
					round_aa += 1
				elif pair == ("B", "B"):
					round_bb += 1
				elif pair == ("A", "B"):
					round_ab += 1

			round_aa_norm = (round_aa / denom_aa) if denom_aa > 0 else 0
			round_bb_norm = (round_bb / denom_bb) if denom_bb > 0 else 0
			round_ab_norm = (round_ab / denom_ab) if denom_ab > 0 else 0

			per_round_details.append({
				"stage": current_stage,
				"prev_stage": prev_stage,
				"prev_tally": {"A": count_a, "B": count_b},
				"A pings A": round_aa,
				"B pings B": round_bb,
				"A-B pings": round_ab,
				"A pings A(normalized)": round_aa_norm,
				"B pings B(normalized)": round_bb_norm,
				"A-B pings(normalized)": round_ab_norm,
			})

			# Accumulate per-problem file totals
			ps_pings_AA += round_aa
			ps_pings_BB += round_bb
			ps_pings_AB += round_ab
			ps_pings_AA_norm += round_aa_norm
			ps_pings_BB_norm += round_bb_norm
			ps_pings_AB_norm += round_ab_norm

		if not ps_is_valid:
			continue
		final_data = {
			"dataset": dataset,
			"pid": pid,
			"identity": identity,
			"agent_roster": agent_roster,
			"per_round_breakdown": per_round_details,
			"A pings A": ps_pings_AA,
			"B pings B": ps_pings_BB,
			"A-B pings": ps_pings_AB,
			"A pings A(normalized)": ps_pings_AA_norm,
			"B pings B(normalized)": ps_pings_BB_norm,
			"A-B pings(normalized)": ps_pings_AB_norm,
		}
		data_set_array.append(final_data)

		data_set_pings_AA += ps_pings_AA
		data_set_pings_BB += ps_pings_BB
		data_set_pings_AB += ps_pings_AB
		data_set_pings_AA_norm += ps_pings_AA_norm
		data_set_pings_BB_norm += ps_pings_BB_norm
		data_set_pings_AB_norm += ps_pings_AB_norm

	experiment_6_filter_2[dataset]["Array"] = data_set_array
	dataset_details = {
		"A pings A": data_set_pings_AA,
		"B pings B": data_set_pings_BB,
		"A-B pings": data_set_pings_AB,
		"A pings A(normalized)": data_set_pings_AA_norm,
		"B pings B(normalized)": data_set_pings_BB_norm,
		"A-B pings(normalized)": data_set_pings_AB_norm,
	}
	experiment_6_filter_2[dataset]["Dataset final"] = dataset_details

	total_pings_AA += data_set_pings_AA
	total_pings_BB += data_set_pings_BB
	total_pings_AB += data_set_pings_AB
	total_pings_AA_norm += data_set_pings_AA_norm
	total_pings_BB_norm += data_set_pings_BB_norm
	total_pings_AB_norm += data_set_pings_AB_norm

experiment_6_filter_2["Final Results"] = {
	"A pings A": total_pings_AA,
	"B pings B": total_pings_BB,
	"A-B pings": total_pings_AB,
	"A pings A(normalized)": total_pings_AA_norm,
	"B pings B(normalized)": total_pings_BB_norm,
	"A-B pings(normalized)": total_pings_AB_norm,
}

with open(experiment_path[5] + "/results_filter_2.json", "w", encoding="utf-8") as f:
	json.dump(experiment_6_filter_2, f, indent=4)

In [ ]:
excel_data = AutoDict()

excel_data["Master analytics"] = {	
	"mmlu": experiment_6_filter_2["mmlu"]["Dataset final"],
	"moralchoice": experiment_6_filter_2["moralchoice"]["Dataset final"],
	"MoralMachineHuman": experiment_6_filter_2["MoralMachineHuman"]["Dataset final"],
	"Final Results": experiment_6_filter_2["Final results"]
}

excel_path = Path(experiment_path[5]+"/result_filter_2.xlsx")

with pd.ExcelWriter(excel_path, engine="openpyxl") as writer:
	for sheets, records in excel_data.items():
		df = pd.DataFrame.from_dict(records, orient="index")
		df.index.name = "Dataset"
		df.reset_index(inplace=True)
		df.to_excel(writer, sheet_name=sheets, index=False)

# Experiment 7

In [ ]:
experiment_7_full_data = AutoDict()

family_combinations = set(x+"+"+y for x in agent_family for y in agent_family)
family_combinations_count = AutoDict()
for family in family_combinations:
	f1, f2 = family.split("+")
	if f1 == f2:
		family_combinations_count[family] = math.comb(agent_family_count[f1], 2)
	else:
		family_combinations_count[family] = agent_family_count[f1] * agent_family_count[f2]
family_family_pings = AutoDict({x:0 for x in family_combinations})
family_family_pings_norm = AutoDict({x:0 for x in family_combinations})

for folder in data_set_folders:
	data_set_ffp = AutoDict({x:0 for x in family_combinations})
	data_set_ffp_norm = AutoDict({x:0 for x in family_combinations})
	data_set_array = []
	if not folder.exists():
		print(f"Directory not found: {folder.name}")
		continue
	for json_file in folder.rglob("*.json"):
		with json_file.open("r", encoding="utf-8") as f:
			content = json.load(f)
		dataset = content["simulation_metadata"]["dataset"].split("/")[1]
		pid = content["simulation_metadata"]["pid"]
		identity = content["simulation_metadata"]["identity_condition"]
		agent_roster = content["simulation_metadata"]["agent_roster"]
		agent_dict = {agent["agent_id"]: {k:v for k, v in agent.items() if k != "agent_id"} for agent in agent_roster}
		rounds = content["stages_detail"][4:15]
		rounds = [x["statements"] for x in rounds]
		speakers = [x[y]["speaker_agent_id"] for x in rounds for y in range(11)]
		targets = [x[y]["target_agent_id"] for x in rounds for y in range(11)]
		ps_ffp = AutoDict({x:0 for x in family_combinations})
		for speaker, target in zip(speakers, targets):
			ps_ffp[agent_dict[speaker]["family"]+"+"+agent_dict[target]["family"]] += 1
		ps_ffp_norm = {f: p/family_combinations_count[f] for f, p in ps_ffp.items()}
		final_data = {
			"dataset": dataset,
			"pid": pid,
			"identity": identity,
			"agent_roster": agent_roster,
			"Family-Family pings": ps_ffp,
			"Family-Family pings(normalized)": ps_ffp_norm
		}
		data_set_array.append(final_data)
		data_set_ffp = {fam: data_set_ffp[fam] + ps_ffp[fam] for fam in data_set_ffp.keys()}
		data_set_ffp_norm = {fam: data_set_ffp_norm[fam] + ps_ffp_norm[fam] for fam in data_set_ffp_norm.keys()}
	experiment_7_full_data[dataset]["Array"] = data_set_array
	dataset_details = {
		"Family-Family Pings": data_set_ffp,
		"Family-Family Pings(normalized)": data_set_ffp_norm
	}
	family_family_pings = {fam: family_family_pings[fam] + data_set_ffp[fam] for fam in family_family_pings.keys()}
	family_family_pings_norm = {fam: family_family_pings_norm[fam] + data_set_ffp_norm[fam] for fam in family_family_pings_norm.keys()}
	experiment_7_full_data[dataset]["Dataset final"] = dataset_details
	pass
experiment_7_full_data["Final Results"] = {
	"Family-Family Pings": family_family_pings,
	"Family-Family Pings(normalized)": family_family_pings_norm
}
experiment_7_full_data["Note"] = "Family-Family is written as Speark+Target. Eg. GemmaLLaMA means Gemma is Speaker and LLaMA is target"

with open(experiment_path[6]+"/results.json","w",encoding="utf-8") as f:
	json.dump(experiment_7_full_data, f, indent=4)

In [ ]:
excel_data = AutoDict()

excel_data["Master analytics"] = experiment_7_full_data["Final Results"]

excel_data["MMLU"] = experiment_7_full_data["mmlu"]["Dataset final"]
excel_data["moralchoice"] = experiment_7_full_data["moralchoice"]["Dataset final"]
excel_data["MoralMachineHuman"] = experiment_7_full_data["MoralMachineHuman"]["Dataset final"]

excel_path = Path(experiment_path[6]+"/result.xlsx")

with pd.ExcelWriter(excel_path, engine="openpyxl") as writer:
	for sheets, records in excel_data.items():
		df = pd.DataFrame.from_dict(records, orient="index")
		df.index.name = "Final Stats"
		df.reset_index(inplace=True)
		df.to_excel(writer, sheet_name=sheets, index=False)

# Experiment 8

In [ ]:
experiment_8_full_data = AutoDict()

total_pings_RR = 0
total_pings_PP = 0
total_pings_RP = 0
total_pings_PR = 0
total_pings_RR_norm = 0
total_pings_PP_norm = 0
total_pings_RP_norm = 0
total_pings_PR_norm = 0

for folder in data_set_folders:
	data_set_pings_RR = 0
	data_set_pings_PP = 0
	data_set_pings_RP = 0
	data_set_pings_PR = 0
	data_set_pings_RR_norm = 0
	data_set_pings_PP_norm = 0
	data_set_pings_RP_norm = 0
	data_set_pings_PR_norm = 0
	data_set_array = []

	if not folder.exists():
		print(f"Directory not found: {folder.name}")
		continue
	for json_file in folder.rglob("*.json"):
		with json_file.open("r", encoding="utf-8") as f:
			content = json.load(f)
			pass
		dataset = content["simulation_metadata"]["dataset"].split("/")[1]
		pid = content["simulation_metadata"]["pid"]
		identity = content["simulation_metadata"]["identity_condition"]
		agent_roster = content["simulation_metadata"]["agent_roster"]
		agent_dict = {agent["agent_id"]: {k:v for k, v in agent.items() if k != "agent_id"} for agent in agent_roster}
		rounds = content["stages_detail"][4:15]
		rounds = [x["statements"] for x in rounds]
		speakers = [x[y]["speaker_agent_id"] for x in rounds for y in range(11)]
		targets = [x[y]["target_agent_id"] for x in rounds for y in range(11)]
		ps_pings_RR = 0
		ps_pings_PP = 0
		ps_pings_RP = 0
		ps_pings_PR = 0
		ps_pings_RR_norm = 0
		ps_pings_PP_norm = 0
		ps_pings_RP_norm = 0
		ps_pings_PR_norm = 0
		for speaker, target in zip(speakers, targets):
			if agent_dict[speaker]["size"] in rich_family:
				if agent_dict[target]["size"] in rich_family:
					ps_pings_RR += 1
					pass
				else:
					ps_pings_RP+=1
					pass
				pass
			else:
				if agent_dict[target]["size"] in rich_family:
					ps_pings_PR += 1
					pass
				else:
					ps_pings_PP += 1
					pass
				pass
			pass
		ps_pings_PP_norm = ps_pings_PP / math.comb(8, 2)
		ps_pings_RR_norm = ps_pings_RR / math.comb(3, 2)
		ps_pings_PR_norm = ps_pings_PR / (3*8)
		ps_pings_RP_norm = ps_pings_RP / (3*8)
		final_data = {
			"dataset": dataset,
			"pid": pid,
			"identity": identity,
			"agent_roster": agent_roster,
			"rich_family": rich_family,
			"Rich pings Rich": ps_pings_RR,
			"Rich pings Poor": ps_pings_RP,
			"Poor pings Rich": ps_pings_PR,
			"Poor pings Poor": ps_pings_PP,
			"Rich pings Rich(normalized)": ps_pings_RR_norm,
			"Rich pings Poor(normalized)": ps_pings_RP_norm,
			"Poor pings Rich(normalized)": ps_pings_PR_norm,
			"Poor pings Poor(normalized)": ps_pings_PP_norm,
		}
		data_set_array.append(final_data)
		data_set_pings_RR += ps_pings_RR
		data_set_pings_PP += ps_pings_PP
		data_set_pings_RP += ps_pings_RP
		data_set_pings_PR += ps_pings_PR
		data_set_pings_RR_norm += ps_pings_RR_norm
		data_set_pings_PP_norm += ps_pings_PP_norm
		data_set_pings_RP_norm += ps_pings_RP_norm
		data_set_pings_PR_norm += ps_pings_PR_norm
	experiment_8_full_data[dataset]["Array"] = data_set_array
	dataset_details = {
		"Rich pings Rich": data_set_pings_RR,
		"Rich pings Poor": data_set_pings_RP,
		"Poor pings Rich": data_set_pings_PR,
		"Poor pings Poor": data_set_pings_PP,
		"Rich pings Rich(normalized)": data_set_pings_RR_norm,
		"Rich pings Poor(normalized)": data_set_pings_RP_norm,
		"Poor pings Rich(normalized)": data_set_pings_PR_norm,
		"Poor pings Poor(normalized)": data_set_pings_PP_norm,
	}
	experiment_8_full_data[dataset]["Dataset final"] = dataset_details
	total_pings_RR += data_set_pings_RR
	total_pings_PP += data_set_pings_PP
	total_pings_RP += data_set_pings_RP
	total_pings_PR += data_set_pings_PR
	total_pings_RR_norm += data_set_pings_RR_norm
	total_pings_PP_norm += data_set_pings_PP_norm
	total_pings_RP_norm += data_set_pings_RP_norm
	total_pings_PR_norm += data_set_pings_PR_norm
experiment_8_full_data["Final Results"] = {
	"Rich pings Rich": total_pings_RR,
	"Rich pings Poor": total_pings_RP,
	"Poor pings Rich": total_pings_PR,
	"Poor pings Poor": total_pings_PP,
	"Rich pings Rich(normalized)": total_pings_RR_norm,
	"Rich pings Poor(normalized)": total_pings_RP_norm,
	"Poor pings Rich(normalized)": total_pings_PR_norm,
	"Poor pings Poor(normalized)": total_pings_PP_norm,
}

with open(experiment_path[7]+"/results.json", "w", encoding="utf-8") as f:
	json.dump(experiment_8_full_data, f, indent=4)

In [ ]:
excel_data = AutoDict()

excel_data["Master analytics"] = experiment_8_full_data["Final Results"]

excel_data["MMLU"] = experiment_8_full_data["mmlu"]["Dataset final"]
excel_data["moralchoice"] = experiment_8_full_data["moralchoice"]["Dataset final"]
excel_data["MoralMachineHuman"] = experiment_8_full_data["MoralMachineHuman"]["Dataset final"]

excel_path = Path(experiment_path[7]+"/result.xlsx")

with pd.ExcelWriter(excel_path, engine="openpyxl") as writer:
	for sheets, records in excel_data.items():
		df = pd.DataFrame.from_dict(records, orient="index")
		df.index.name = sheets
		df.reset_index(inplace=True)
		df.to_excel(writer, sheet_name=sheets, index=False)

# Experiment 9

In [ ]:
experiment_9_full_data = AutoDict()

total_MajRational = 0
total_MinRational = 0

for folder in data_set_folders:
	data_set_MajRational = 0
	data_set_MinRational = 0
	data_set_array = []

	if not folder.exists():
		print(f"Directory not found: {folder.name}")
		continue
	for json_file in folder.rglob("*.json"):
		with json_file.open("r", encoding="utf-8") as f:
			content = json.load(f)
			pass
		dataset = content["simulation_metadata"]["dataset"].split("/")[1]
		pid = content["simulation_metadata"]["pid"]
		identity = content["simulation_metadata"]["identity_condition"]
		agent_roster = content["simulation_metadata"]["agent_roster"]
		round_1 = content["stages_detail"][1]
		round_3 = content["stages_detail"][3]
		majority = "A" if round_1["tally"]["A"] > round_1["tally"]["B"] else "B"
		minority = "A" if round_1["tally"]["A"] < round_1["tally"]["B"] else "B"
		MajRational = 0
		MinRational = 0
		flip_agents = []
		for agent in round_3["votes"]:
			if agent["previous_vote"] != agent["vote"]:
				if agent["vote"] == majority:
					agent["flip"] = "majority"
					MajRational+=1
				else:
					agent["flip"] = "minority"
					MinRational+=1
				flip_agents.append(agent)
		final_data = {
			"dataset": dataset,
			"pid": pid,
			"identity": identity,
			"agent_roster": agent_roster,
			"round_3 details": round_3,
			"Majority": majority,
			"Minority": minority,
			"flipped": flip_agents,
			"Majority Rational Bias": MajRational,
			"Minority Rational Bias": MinRational
		}
		data_set_array.append(final_data)
		data_set_MajRational += MajRational
		data_set_MinRational += MinRational
	experiment_9_full_data[dataset]["Array"] = data_set_array
	dataset_details = {
		"Majority Rational Bias": data_set_MajRational,
		"Minority Rational Bias": data_set_MinRational
	}
	total_MajRational += data_set_MajRational
	total_MinRational += data_set_MinRational
	experiment_9_full_data[dataset]["Dataset final"] = dataset_details
	pass
experiment_9_full_data["Final results"] = {
	"Majority Rational Bias": total_MajRational,
	"Minority Rational Bias": total_MinRational
}

with open(experiment_path[8]+"/results.json", "w", encoding="utf-8") as f:
	json.dump(experiment_9_full_data, f, indent=4)

In [ ]:
excel_data = AutoDict()
# excel_data["mmlu"] = None
# excel_data["moralchoice"] = None
# excel_data["MoralMachineHuman"] = None

excel_data["Master analytics"] = {
	"mmlu": experiment_9_full_data["mmlu"]["Dataset final"],
	"moralchoice": experiment_9_full_data["moralchoice"]["Dataset final"],
	"MoralMachineHuman": experiment_9_full_data["MoralMachineHuman"]["Dataset final"],
	"Final Results": experiment_9_full_data["Final results"]
}

excel_path = Path(experiment_path[8]+"/result.xlsx")

with pd.ExcelWriter(excel_path, engine="openpyxl") as writer:
	for sheets, records in excel_data.items():
		df = pd.DataFrame.from_dict(records, orient="index")
		df.index.name = "Dataset"
		df.reset_index(inplace=True)
		df.to_excel(writer, sheet_name=sheets, index=False)
excel_data

# Experiment 10

In [ ]:
experiment_10_full_data = AutoDict()

total_BtOR = 0
total_exceptional = 0

for folder in data_set_folders:
	data_set_BtOR = 0
	data_set_exceptional = 0
	data_set_array = []

	if not folder.exists():
		print(f"Directory not found: {folder.name}")
		continue
	for json_file in folder.rglob("*.json"):
		with json_file.open("r", encoding="utf-8") as f:
			content = json.load(f)
			pass
		dataset = content["simulation_metadata"]["dataset"].split("/")[1]
		pid = content["simulation_metadata"]["pid"]
		identity = content["simulation_metadata"]["identity_condition"]
		agent_roster = content["simulation_metadata"]["agent_roster"]
		round_0 = content["stages_detail"][0]
		round_1 = content["stages_detail"][1]
		round_3 = content["stages_detail"][3]
		agent_groups = {r0["agent_id"]:[r0, r1,r3] for r0 in round_0["votes"] for r1 in round_1["votes"] for r3 in round_3["votes"] if r0["agent_id"] ==  r1["agent_id"] == r3["agent_id"]}
		BtOR = 0
		exceptional=0
		flip_agents = []
		for agent_r0, agent_r1, agent_r3 in agent_groups.values():
			if agent_r3["vote"] != agent_r1["vote"]:
				if agent_r3["vote"] == agent_r0["vote"]:
					agent_r3["flip"] = "Back to Own Rational"
					BtOR+=1
				else:
					agent_r3["flip"] = "Exceptional"
					exceptional+=1
				flip_agents.append({"r0":agent_r0, "r1":agent_r1, "r2":agent_r3})
		final_data = {
			"dataset": dataset,
			"pid": pid,
			"identity": identity,
			"agent_roster": agent_roster,
			"round_0 details": round_0,
			"round_1 details": round_1,
			"round_3 details": round_3,
			"flipped": flip_agents,
			"Back to Own Rational": BtOR,
			"Exceptional": exceptional
		}
		data_set_array.append(final_data)
		data_set_BtOR += BtOR
		data_set_exceptional += exceptional
	experiment_10_full_data[dataset]["Array"] = data_set_array
	dataset_details = {
		"Back to Own Rational": data_set_BtOR,
		"Exceptional": data_set_exceptional
	}
	total_BtOR += data_set_BtOR
	total_exceptional += data_set_exceptional
	experiment_10_full_data[dataset]["Dataset final"] = dataset_details
	pass
experiment_10_full_data["Final results"] = {
	"Back to Own Rational": total_BtOR,
	"Exceptional": total_exceptional
}

with open(experiment_path[9]+"/results.json", "w", encoding="utf-8") as f:
	json.dump(experiment_10_full_data, f, indent=4)

In [ ]:
excel_data = AutoDict()
# excel_data["mmlu"] = None
# excel_data["moralchoice"] = None
# excel_data["MoralMachineHuman"] = None

excel_data["Master analytics"] = {
	"mmlu": experiment_10_full_data["mmlu"]["Dataset final"],
	"moralchoice": experiment_10_full_data["moralchoice"]["Dataset final"],
	"MoralMachineHuman": experiment_10_full_data["MoralMachineHuman"]["Dataset final"],
	"Final Results": experiment_10_full_data["Final results"]
}

excel_path = Path(experiment_path[9]+"/result.xlsx")

with pd.ExcelWriter(excel_path, engine="openpyxl") as writer:
	for sheets, records in excel_data.items():
		df = pd.DataFrame.from_dict(records, orient="index")
		df.index.name = "Dataset"
		df.reset_index(inplace=True)
		df.to_excel(writer, sheet_name=sheets, index=False)
excel_data

In [ ]:
experiment_10_2_full_data = AutoDict()

total_MfB_1 = 0
total_MR_1 = 0
total_exceptional = 0

for folder in data_set_folders:
	data_set_MfB_1 = 0
	data_set_MR_1 = 0
	data_set_exceptional = 0
	data_set_array = []
	if not folder.exists():
		print(f"Directory not found: {folder.name}")
		continue
	for json_file in folder.rglob("*.json"):
		with json_file.open("r", encoding="utf-8") as f:
			content = json.load(f)
			pass
		dataset = content["simulation_metadata"]["dataset"].split("/")[1]
		pid = content["simulation_metadata"]["pid"]
		identity = content["simulation_metadata"]["identity_condition"]
		agent_roster = content["simulation_metadata"]["agent_roster"]
		round_3 = content["stages_detail"][3]
		round_1 = content["stages_detail"][1]["tally"]
		r1_Maj = "A" if round_1["A"] > round_1["B"] else "B"
		r1_min = "A" if round_1["A"] < round_1["B"] else "B"
		family_stats = AutoDict({x : {"r1_A": 0, "r1_B": 0, "r3_A":0, "r3_B":0,"Majority": None, "Minority": None} for x in agent_family})
		rich_poor_stats = AutoDict({"rich": {"r1_A": 0, "r1_B": 0, "r3_A":0, "r3_B":0,"Majority": None, "Minority": None}, "poor": {"r1_A": 0, "r1_B": 0, "r3_A":0, "r3_B":0,"Majority": None, "Minority": None}})
		MfB_1 = 0
		MR_1 = 0
		exceptional = 0
		flip_agents = []
		for agent in round_3["votes"]:
			family_stats[agent["family"]]["r1_" +agent["previous_vote"]] += 1
			family_stats[agent["family"]]["r3_"+agent["vote"]] +=1
			rich_poor_stats["rich" if agent["size_str"] in rich_family else "poor"]["r1_"+agent["previous_vote"]] += 1
			rich_poor_stats["rich" if agent["size_str"] in rich_family else "poor"]["r3_"+agent["vote"]] +=1
		for family_name, family in family_stats.items():
			if family["r1_A"] > family["r1_B"]:
				family["Majority"], family["Minority"] = "A", "B"
			elif family["r1_A"] < family["r1_B"]:
				family["Minority"], family["Majority"] = "A", "B"
			else:
				if family["r3_A"] > family["r3_B"]:
					family["Majority"], family["Minority"] = "A", "B"
				elif family["r3_A"] < family["r3_B"]:
					family["Minority"], family["Majority"] = "A", "B"
				else:
					family["Majority"], family["Minority"] = "TIE", "TIE"
				pass
		for type_name, type in  rich_poor_stats.items():
			if type["r1_A"] > type["r1_B"]:
				type["Majority"], type["Minority"] = "A", "B"
			elif type["r1_A"] < type["r1_B"]:
				type["Minority"], type["Majority"] = "A", "B"
			else:
				if type["r3_A"] > type["r3_B"]:
					type["Majority"], type["Minority"] = "A", "B"
				elif type["r3_A"] < type["r3_B"]:
					type["Minority"], type["Majority"] = "A", "B"
				else:
					type["Majority"], type["Minority"] = "TIE", "TIE"
				pass
		for agent in round_3["votes"]:
			is_exceptional = [False, False]
			if agent["previous_vote"] != agent["vote"] and agent["vote"] == r1_min:
				if family_stats[agent["family"]]["Majority"] == agent["vote"]:
					MfB_1 += 1
				else:
					is_exceptional[0] = True
				if rich_poor_stats["rich" if agent["size_str"] in rich_family else "poor"]["Majority"] == agent["vote"]:
					MR_1 += 1
				else:
					is_exceptional[1] = True
				if False not in is_exceptional:
					exceptional += 1
				flip_agents.append(agent)
		final_data = {
			"dataset": dataset,
			"pid": pid,
			"identity": identity,
			"agent_roster": agent_roster,
			"round_3 details": round_3,
			"rich_family": rich_family,
			"family_stats": family_stats,
			"rich_poor_stats": rich_poor_stats,
			"flipped": flip_agents,
			"Majority": r1_Maj,
			"Minority": r1_min,
			"Majority family bias": MfB_1,
			"Majority rich bias": MR_1,
			"Exceptional": exceptional
		}
		data_set_array.append(final_data)
		data_set_MfB_1+=MfB_1
		data_set_MR_1+= MR_1
		data_set_exceptional+=exceptional
	experiment_10_2_full_data[dataset]["Array"] = data_set_array
	dataset_details = {
		"Majority family Bias": data_set_MfB_1,
		"Majority rich Bias": data_set_MR_1,
		"Exceptional": data_set_exceptional
	}
	total_MfB_1 += data_set_MfB_1
	total_MR_1 += data_set_MR_1
	total_exceptional += data_set_exceptional
	experiment_10_2_full_data[dataset]["Dataset final"] = dataset_details
	pass
experiment_10_2_full_data["Final results"] = {
	"Majority family Bias": total_MfB_1,
	"Majority rich Bias": total_MR_1,
	"Exceptional": total_exceptional
}

with open(experiment_path[9]+"/results_2.json","w",encoding="utf-8") as f:
	json.dump(experiment_10_2_full_data, f, indent=4)

# Experiment 11

In [ ]:
experiment_11_full_data = AutoDict()

total_BtORM = 0
total_BtORMi = 0

for folder in data_set_folders:
	data_set_BtORM = 0
	data_set_BtORMi = 0
	data_set_array = []

	if not folder.exists():
		print(f"Directory not found: {folder.name}")
		continue
	for json_file in folder.rglob("*.json"):
		with json_file.open("r", encoding="utf-8") as f:
			content = json.load(f)
			pass
		dataset = content["simulation_metadata"]["dataset"].split("/")[1]
		pid = content["simulation_metadata"]["pid"]
		identity = content["simulation_metadata"]["identity_condition"]
		agent_roster = content["simulation_metadata"]["agent_roster"]
		round_0 = content["stages_detail"][0]
		round_1 = content["stages_detail"][1]
		majority = "A" if round_1["tally"]["A"] > round_1["tally"]["B"] else "B"
		minority = "A" if round_1["tally"]["A"] < round_1["tally"]["B"] else "B"
		round_3 = content["stages_detail"][3]
		agent_groups = {r0["agent_id"]:[r0, r1,r3] for r0 in round_0["votes"] for r1 in round_1["votes"] for r3 in round_3["votes"] if r0["agent_id"] ==  r1["agent_id"] == r3["agent_id"]}
		BtORM = 0
		BtORMi=0
		flip_agents = []
		for agent_r0, agent_r1, agent_r3 in agent_groups.values():
			if agent_r3["vote"] != agent_r1["vote"] and agent_r3["vote"] == agent_r0["vote"]:
				if agent_r3["vote"] == majority:
					agent_r3["flip"] = "Back to Own Rational Majority"
					BtORM+=1
				else:
					agent_r3["flip"] = "Back to Own Rational Minority"
					BtORMi+=1
				flip_agents.append({"r0":agent_r0, "r1":agent_r1, "r2":agent_r3})
		final_data = {
			"dataset": dataset,
			"pid": pid,
			"identity": identity,
			"agent_roster": agent_roster,
			"round_0 details": round_0,
			"round_1 details": round_1,
			"round_3 details": round_3,
			"flipped": flip_agents,
			"Back to Own Rational Majority": BtORM,
			"Back to Own Rational Minority": BtORMi
		}
		data_set_array.append(final_data)
		data_set_BtORM += BtORM
		data_set_BtORMi += BtORMi
	experiment_11_full_data[dataset]["Array"] = data_set_array
	dataset_details = {
		"Back to Own Rational Majority": data_set_BtORM,
		"Back to Own Rational Minority": data_set_BtORMi
	}
	total_BtORM += data_set_BtORM
	total_BtORMi += data_set_BtORMi
	experiment_11_full_data[dataset]["Dataset final"] = dataset_details
	pass
experiment_11_full_data["Final results"] = {
	"Back to Own Rational Majority": total_BtORM,
	"Back to Own Rational Minority": total_BtORMi
}

with open(experiment_path[10]+"/results.json", "w", encoding="utf-8") as f:
	json.dump(experiment_11_full_data, f, indent=4)

In [ ]:
excel_data = AutoDict()
# excel_data["mmlu"] = None
# excel_data["moralchoice"] = None
# excel_data["MoralMachineHuman"] = None

excel_data["Master analytics"] = {
	"mmlu": experiment_11_full_data["mmlu"]["Dataset final"],
	"moralchoice": experiment_11_full_data["moralchoice"]["Dataset final"],
	"MoralMachineHuman": experiment_11_full_data["MoralMachineHuman"]["Dataset final"],
	"Final Results": experiment_11_full_data["Final results"]
}

excel_path = Path(experiment_path[10]+"/result.xlsx")

with pd.ExcelWriter(excel_path, engine="openpyxl") as writer:
	for sheets, records in excel_data.items():
		df = pd.DataFrame.from_dict(records, orient="index")
		df.index.name = "Dataset"
		df.reset_index(inplace=True)
		df.to_excel(writer, sheet_name=sheets, index=False)
excel_data

# Experiment 12

In [ ]:
experiment_12_full_data = AutoDict()

total_talks = {i: 0 for i in range(12)}
total_pings = {i: 0 for i in range(12)}

for folder in data_set_folders:
	data_set_talks = {i: 0 for i in range(12)}
	data_set_pings = {i: 0 for i in range(12)}
	data_set_array = []
	if not folder.exists():
		print(f"Directory not found: {folder.name}")
		continue
	for json_file in folder.rglob("*.json"):
		with json_file.open("r", encoding="utf-8") as f:
			content = json.load(f)
			pass
		dataset = content["simulation_metadata"]["dataset"].split("/")[1]
		pid = content["simulation_metadata"]["pid"]
		identity = content["simulation_metadata"]["identity_condition"]
		agent_roster = content["simulation_metadata"]["agent_roster"]
		stage_statements = {stage["stage_index"]: stage["statements"] for stage in content["stages_detail"][4:15]}
		stage_replies = {stage["stage_index"]: stage["replies"] for stage in content["stages_detail"][4:15]}
		talks = {i:0 for i in range(12)}
		pings = {i:0 for i in range(12)}
		agents = content["summary_analytics"]["agent_trajectories"]
		agent_votes = {agnt["agent_id"]: agnt["vote_history"] for agnt in agents}
		agent_vote_A = {agnt["agent_id"]: 0 for agnt in agents}
		for agent, votes in agent_votes.items():
			count_A = 0
			for i in range(11):
				if votes[i+4] == "A":
					count_A += 1
			agent_vote_A[agent] = count_A
		for stage_index, statements in stage_statements.items():
			for statement in statements:
				pings[abs(agent_vote_A[statement["speaker_agent_id"]] - agent_vote_A[statement["target_agent_id"]])] += 1
		for stage_index, replies in stage_replies.items():
			for reply in replies:
				if reply["decision"] == "reply":
					pings[abs(agent_vote_A[reply["replier_agent_id"]] - agent_vote_A[reply["sender_agent_id"]])] -= 1
					talks[abs(agent_vote_A[reply["replier_agent_id"]] - agent_vote_A[reply["sender_agent_id"]])] += 1
		final_data = {
			"dataset": dataset,
			"pid": pid,
			"identity": identity,
			"agent_roster": agent_roster,
			"Pings": pings,
			"Talks": talks
		}
		data_set_array.append(final_data)
		for k, v in pings.items():
			data_set_pings[k] += v
		for k, v in talks.items():
			data_set_talks[k] += v
	experiment_12_full_data[dataset]["Array"] = data_set_array
	dataset_details = {
		"Pings": data_set_pings,
		"Talks": data_set_talks
	}
	for k, v in data_set_pings.items():
		total_pings[k] += v
	for k, v in data_set_talks.items():
		total_talks[k] += v
	experiment_12_full_data[dataset]["Dataset final"] = dataset_details
	pass
experiment_12_full_data["Final results"] = {
	"Pings": total_pings,
	"Talks": total_talks
}

with open(experiment_path[11]+"/results.json", "w", encoding="utf-8") as f:
	json.dump(experiment_12_full_data, f, indent=4)

In [ ]:
excel_data = AutoDict()
# excel_data["mmlu"] = None
# excel_data["moralchoice"] = None
# excel_data["MoralMachineHuman"] = None

excel_data["Master analytics"] = experiment_12_full_data["Final results"]
excel_data["mmlu"] = experiment_12_full_data["mmlu"]["Dataset final"]
excel_data["moralchoice"] = experiment_12_full_data["moralchoice"]["Dataset final"]
excel_data["MoralMachineHuman"] = experiment_12_full_data["MoralMachineHuman"]["Dataset final"]

excel_path = Path(experiment_path[11]+"/result.xlsx")

with pd.ExcelWriter(excel_path, engine="openpyxl") as writer:
	for sheets, records in excel_data.items():
		df = pd.DataFrame.from_dict(records, orient="index")
		df.index.name = "Communication"
		df.reset_index(inplace=True)
		df.to_excel(writer, sheet_name=sheets, index=False)
excel_data

In [ ]:
filter_results_exp12 = AutoDict()

total_pid_count = 0

for folder in data_set_folders:
	data_set_pid = []
	data_set_pid_count = 0
	if not folder.exists():
		print(f"Directory not found: {folder.name}")
		continue
	for json_file in folder.rglob("*.json"):
		with json_file.open("r", encoding="utf-8") as f:
			content = json.load(f)

		dataset = content["simulation_metadata"]["dataset"].split("/")[1]
		pid = content["simulation_metadata"]["pid"]
		agents = content["summary_analytics"]["agent_trajectories"]
		always_A = []
		always_B = []
		flip_once = []
		flip_more = []
		agent_votes = {agnt["agent_id"]: agnt["vote_history"] for agnt in agents}
		for agent, votes in agent_votes.items():
			flip_count = 0
			for i in range(1,12):
				flip_count += 1 if votes[i-1] != votes[i] else 0
				pass
			if flip_count == 0:
				if votes[0] == "A":
					always_A.append(agent)
				else:
					always_B.append(agent)
			else:
				if flip_count == 1:
					flip_once.append(agent)
				else:
					flip_more.append(agent)
		if len(always_A) >= 2 and len(always_B) >= 2:
			data_set_pid.append(pid)
			data_set_pid_count+=1
	total_pid_count += data_set_pid_count
	filter_results_exp12[dataset]["Filtered questions"] = {
		"Array": data_set_pid,
		"Count": data_set_pid_count
	}
filter_results_exp12["Final count"] = total_pid_count

with open(experiment_path[11]+"/filtered.json", "w", encoding="utf-8") as f:
	json.dump(filter_results_exp12, f, indent=4)

In [ ]:
experiment_12_filter = AutoDict()

total_talks = {i: 0 for i in range(12)}
total_pings = {i: 0 for i in range(12)}

for folder in data_set_folders:
	data_set_talks = {i: 0 for i in range(12)}
	data_set_pings = {i: 0 for i in range(12)}
	data_set_array = []
	if not folder.exists():
		print(f"Directory not found: {folder.name}")
		continue
	for json_file in folder.rglob("*.json"):
		with json_file.open("r", encoding="utf-8") as f:
			content = json.load(f)
			pass
		dataset = content["simulation_metadata"]["dataset"].split("/")[1]
		pid = content["simulation_metadata"]["pid"]
		if not pid in filter_results_exp12[dataset]["Filtered questions"]["Array"]:
			continue
		identity = content["simulation_metadata"]["identity_condition"]
		agent_roster = content["simulation_metadata"]["agent_roster"]
		stage_statements = {stage["stage_index"]: stage["statements"] for stage in content["stages_detail"][4:15]}
		stage_replies = {stage["stage_index"]: stage["replies"] for stage in content["stages_detail"][4:15]}
		talks = {i:0 for i in range(12)}
		pings = {i:0 for i in range(12)}
		agents = content["summary_analytics"]["agent_trajectories"]
		agent_votes = {agnt["agent_id"]: agnt["vote_history"] for agnt in agents}
		agent_vote_A = {agnt["agent_id"]: 0 for agnt in agents}
		for agent, votes in agent_votes.items():
			count_A = 0
			for i in range(11):
				if votes[i+4] == "A":
					count_A += 1
			agent_vote_A[agent] = count_A
		for stage_index, statements in stage_statements.items():
			for statement in statements:
				pings[abs(agent_vote_A[statement["speaker_agent_id"]] - agent_vote_A[statement["target_agent_id"]])] += 1
		for stage_index, replies in stage_replies.items():
			for reply in replies:
				if reply["decision"] == "reply":
					pings[abs(agent_vote_A[reply["replier_agent_id"]] - agent_vote_A[reply["sender_agent_id"]])] -= 1
					talks[abs(agent_vote_A[reply["replier_agent_id"]] - agent_vote_A[reply["sender_agent_id"]])] += 1
		final_data = {
			"dataset": dataset,
			"pid": pid,
			"identity": identity,
			"agent_roster": agent_roster,
			"Pings": pings,
			"Talks": talks
		}
		data_set_array.append(final_data)
		for k, v in pings.items():
			data_set_pings[k] += v
		for k, v in talks.items():
			data_set_talks[k] += v
	experiment_12_filter[dataset]["Array"] = data_set_array
	dataset_details = {
		"Pings": data_set_pings,
		"Talks": data_set_talks
	}
	for k, v in data_set_pings.items():
		total_pings[k] += v
	for k, v in data_set_talks.items():
		total_talks[k] += v
	experiment_12_filter[dataset]["Dataset final"] = dataset_details
	pass
experiment_12_filter["Final results"] = {
	"Pings": total_pings,
	"Talks": total_talks
}

with open(experiment_path[11]+"/results_filtered.json", "w", encoding="utf-8") as f:
	json.dump(experiment_12_filter, f, indent=4)

In [ ]:
excel_data = AutoDict()
# excel_data["mmlu"] = None
# excel_data["moralchoice"] = None
# excel_data["MoralMachineHuman"] = None

excel_data["Master analytics"] = experiment_12_filter["Final results"]
excel_data["mmlu"] = experiment_12_filter["mmlu"]["Dataset final"]
excel_data["moralchoice"] = experiment_12_filter["moralchoice"]["Dataset final"]
excel_data["MoralMachineHuman"] = experiment_12_filter["MoralMachineHuman"]["Dataset final"]

excel_path = Path(experiment_path[11]+"/result_filtered.xlsx")
	
with pd.ExcelWriter(excel_path, engine="openpyxl") as writer:
	for sheets, records in excel_data.items():
		df = pd.DataFrame.from_dict(records, orient="index")
		df.index.name = "Communication"
		df.reset_index(inplace=True)
		df.to_excel(writer, sheet_name=sheets, index=False)
excel_data